# Moteur de decision - Recommandations operationnelles

## Objectif

Cette notebook ne cherche pas a predire avec precision : elle demontre comment transformer les donnees de stock/demande deja disponibles en **recommandations lisibles et priorisees** pour un decideur non-technique.

Le moteur repose sur des regles metier transparentes (seuils sur le ratio de couverture), pas sur un modele ML. La logique vit dans [`src/decision_engine.py`](src/decision_engine.py) pour etre reutilisable telle quelle dans le futur tableau de bord.

Etapes :
1. Charger les donnees et se placer sur le dernier mois disponible
2. Calculer le risque de penurie par region x groupe sanguin
3. Valider le moteur contre les etiquettes historiques du dataset
4. Identifier les opportunites de redistribution entre regions
5. Produire une liste de recommandations priorisees, pretes a presenter

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd()))
from src import decision_engine as de

data_path = Path.cwd() / "cnts_simulated_blood_demand.csv"
df = pd.read_csv(data_path)

latest_month = df["month"].max()
latest_month

'2025-12'

## 1. Scoring du risque sur le dernier mois

On calcule notre propre `risk_level_calc` a partir des colonnes brutes (stock, demande) plutot que d'utiliser la colonne `risk_level` deja presente dans le dataset simule. C'est volontaire : sur de vraies donnees CNTS, cette etiquette n'existera pas — il faut la calculer nous-memes.

In [2]:
snapshot, transfers = de.build_recommendations(df, latest_month)

cols = ["region", "blood_group", "hospital_demand_units", "available_stock_units",
        "coverage_ratio_calc", "risk_level_calc", "action"]
snapshot[cols].head(15)

,region,blood_group,hospital_demand_units,available_stock_units,coverage_ratio_calc,risk_level_calc,action
0,Bouaké,O+,243,166,0.68,Eleve,Collecte ciblee urgente + verifier une redistr...
1,Abidjan,B+,370,315,0.85,Eleve,Collecte ciblee urgente + verifier une redistr...
2,Yamoussoukro,O+,162,117,0.72,Eleve,Collecte ciblee urgente + verifier une redistr...
3,San Pedro,O+,155,118,0.76,Eleve,Collecte ciblee urgente + verifier une redistr...
4,Abidjan,A-,83,48,0.58,Eleve,Collecte ciblee urgente + verifier une redistr...
5,Korhogo,O+,123,88,0.72,Eleve,Collecte ciblee urgente + verifier une redistr...
6,Yamoussoukro,A+,120,92,0.77,Eleve,Collecte ciblee urgente + verifier une redistr...
7,Korhogo,B+,64,39,0.61,Eleve,Collecte ciblee urgente + verifier une redistr...
8,Abidjan,O-,78,54,0.69,Eleve,Collecte ciblee urgente + verifier une redistr...
9,Yamoussoukro,B+,90,66,0.73,Eleve,Collecte ciblee urgente + verifier une redistr...


## 2. Validation : le moteur reproduit-il le risque historique ?

Le dataset simule contient deja une colonne `risk_level` (utilisee pour generer les donnees). On compare notre `risk_level_calc`, calcule uniquement a partir de `available_stock_units` et `hospital_demand_units`, a cette etiquette sur l'ensemble des 36 mois. Un fort taux d'accord montre que la regle est coherente avec la logique metier deja en place — un argument utile pour le pitch.

In [3]:
full = df.copy()
full["coverage_ratio_calc"] = de.compute_coverage_ratio(full)
full["risk_level_calc"] = de.classify_risk(full["coverage_ratio_calc"])

accent_map = {"Élevé": "Eleve", "Moyen": "Moyen", "Faible": "Faible"}
full["risk_level_normalized"] = full["risk_level"].map(accent_map)

agreement = (full["risk_level_calc"].astype(str) == full["risk_level_normalized"]).mean()
print(f"Taux d'accord moteur vs historique : {agreement:.1%}")

pd.crosstab(full["risk_level_normalized"], full["risk_level_calc"])

Taux d'accord moteur vs historique : 98.9%


risk_level_calc,Eleve,Moyen,Faible
risk_level_normalized,,,
Eleve,899,0,0
Faible,0,9,162
Moyen,7,363,0


## 3. Opportunites de redistribution

Pour chaque groupe sanguin, on associe les regions en deficit severe (risque Eleve) a des regions en surplus (`coverage_ratio >= 1.3`) du meme groupe. C'est une premiere approximation volontairement simple : elle ignore la compatibilite croisee ABO/Rh (ex. O- donneur universel), a affiner en phase mission avec les equipes medicales.

In [4]:
transfers.sort_values("units_suggested", ascending=False)

,blood_group,from_region,to_region,units_suggested


Sur le dernier mois (2025-12), aucune region n'a de surplus mobilisable (`coverage_ratio >= 1.3`) : la tension est generalisee, pas localisee. C'est en soi un signal utile — pas d'arbitrage de redistribution possible ce mois-ci, la priorite est la collecte. A titre d'illustration, voici un mois ou le moteur detecte bien des transferts (2024-06).

In [5]:
example_snapshot, example_transfers = de.build_recommendations(df, "2024-06")
example_transfers.sort_values("units_suggested", ascending=False)

,blood_group,from_region,to_region,units_suggested
0,AB+,Abidjan,Bouaké,5
1,AB+,Abidjan,Yamoussoukro,5
2,AB+,Abidjan,Korhogo,5


## 4. Recommandations priorisees, pretes a presenter

On assemble le risque et les transferts suggeres en phrases lisibles par un non-technique — c'est ce type de sortie qui doit alimenter le futur tableau de bord.

In [6]:
recommendations = de.build_recommendation_texts(snapshot, transfers)

for line in recommendations[:10]:
    print("-", line)

- Bouaké / O+ — risque Eleve (couverture 0.68). Collecte ciblee urgente + verifier une redistribution depuis une region en surplus.
- Abidjan / B+ — risque Eleve (couverture 0.85). Collecte ciblee urgente + verifier une redistribution depuis une region en surplus.
- Yamoussoukro / O+ — risque Eleve (couverture 0.72). Collecte ciblee urgente + verifier une redistribution depuis une region en surplus.
- San Pedro / O+ — risque Eleve (couverture 0.76). Collecte ciblee urgente + verifier une redistribution depuis une region en surplus.
- Abidjan / A- — risque Eleve (couverture 0.58). Collecte ciblee urgente + verifier une redistribution depuis une region en surplus.
- Korhogo / O+ — risque Eleve (couverture 0.72). Collecte ciblee urgente + verifier une redistribution depuis une region en surplus.
- Yamoussoukro / A+ — risque Eleve (couverture 0.77). Collecte ciblee urgente + verifier une redistribution depuis une region en surplus.
- Korhogo / B+ — risque Eleve (couverture 0.61). Collecte 

## 5. Ce que ça démontre

- Le moteur ne fait que 3 choses : classer le risque, detecter des surplus/deficits compatibles, et les traduire en phrases. Aucune boite noire — chaque recommandation est tracable jusqu'a un seuil explicite.
- La logique vit dans `src/decision_engine.py`, reutilisable telle quelle par un futur tableau de bord (etape suivante).
- Prochaine etape : ajouter une couche de projection simple (moyenne mobile) pour passer de "risque ce mois-ci" a "risque dans les 30 prochains jours", sans sur-promettre la precision.